# Task 2: Exploratory Data Analysis (EDA)
**WeIntern Pvt Ltd - Data Science Week 2 Internship**

**Objective:** Perform comprehensive EDA on the cleaned Superstore sales dataset to uncover patterns, trends, correlations, and actionable business insights.

**Dataset:** cleaned_superstore_sales.csv (output of Task 1)

## 1. Load and Inspect the Cleaned Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', 25)

df = pd.read_csv('cleaned_superstore_sales.csv')
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
df.head()

## 2. Descriptive Statistics

In [ ]:
print('=== Numeric Columns Descriptive Stats ===')
display(df.describe())

print('\n=== Categorical Columns Value Counts (Top) ===')
cat_cols = ['order_priority', 'ship_mode', 'region', 'customer_segment', 
            'product_category', 'product_sub_category']
for col in cat_cols:
    print(f'\n{col}:')
    print(df[col].value_counts().head(8))

## 3. Distributions – Histograms & KDE Plots
Analyze the distribution of key numeric features.

In [ ]:
num_cols = ['sales', 'profit', 'discount', 'unit_price', 'shipping_cost', 'order_quantity']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='steelblue', bins=40)
    axes[i].set_title(f'Distribution of {col}', fontsize=12)
    axes[i].set_xlabel(col)
plt.suptitle('Histograms + KDE for Key Numeric Features', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## 4. Correlation Matrix Heatmap

In [ ]:
corr_cols = ['sales', 'profit', 'discount', 'unit_price', 'shipping_cost', 
             'order_quantity', 'product_base_margin']
corr = df[corr_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdYlBu_r', center=0, fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Numeric Features', fontsize=14)
plt.tight_layout()
plt.show()

print('Strongest positive correlations:')
print(corr.unstack().sort_values(ascending=False).drop_duplicates().head(8))

## 5. Box Plots – Distribution by Category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df, x='product_category', y='sales', ax=axes[0], palette='Set2')
axes[0].set_title('Sales Distribution by Product Category')
axes[0].tick_params(axis='x', rotation=15)

sns.boxplot(data=df, x='region', y='profit', ax=axes[1], palette='Set3')
axes[1].set_title('Profit Distribution by Region')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 6. Trends Over Time (Line Charts)

In [ ]:
# Monthly sales trend
df['year_month'] = df['order_date'].dt.to_period('M')
monthly = df.groupby('year_month').agg({'sales': 'sum', 'profit': 'sum', 'order_id': 'count'}).reset_index()
monthly['year_month'] = monthly['year_month'].astype(str)

fig, ax1 = plt.subplots(figsize=(14, 6))
ax1.plot(monthly['year_month'], monthly['sales'], marker='o', color='royalblue', label='Sales')
ax1.set_xlabel('Year-Month')
ax1.set_ylabel('Total Sales', color='royalblue')
ax1.tick_params(axis='x', rotation=45)
ax1.tick_params(axis='y', labelcolor='royalblue')

ax2 = ax1.twinx()
ax2.plot(monthly['year_month'], monthly['profit'], marker='s', color='green', label='Profit')
ax2.set_ylabel('Total Profit', color='green')
ax2.tick_params(axis='y', labelcolor='green')

plt.title('Monthly Sales & Profit Trend (2009-2012)', fontsize=14)
fig.tight_layout()
plt.show()

print('Peak sales month:', monthly.loc[monthly['sales'].idxmax(), 'year_month'])

## 7. Group-by Analysis on Categorical Variables

In [ ]:
# Sales & Profit by Category and Region
cat_region = df.groupby(['product_category', 'region']).agg({
    'sales': 'sum',
    'profit': 'sum',
    'order_id': 'count'
}).rename(columns={'order_id': 'orders'}).reset_index()

print('=== Sales by Product Category & Region ===')
display(cat_region.pivot(index='product_category', columns='region', values='sales').round(0))

# Customer Segment performance
seg = df.groupby('customer_segment').agg({
    'sales': ['sum', 'mean'],
    'profit': ['sum', 'mean'],
    'order_id': 'count'
}).round(2)
print('\n=== Performance by Customer Segment ===')
display(seg)

## 8. Scatter Plots – Relationships

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.scatterplot(data=df, x='discount', y='profit', hue='product_category', 
                alpha=0.6, ax=axes[0], palette='deep')
axes[0].set_title('Discount vs Profit by Category')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)

sns.scatterplot(data=df, x='unit_price', y='sales', hue='product_category',
                alpha=0.5, ax=axes[1], palette='muted')
axes[1].set_title('Unit Price vs Sales by Category')

plt.tight_layout()
plt.show()

## 9. Key Data-Driven Insights (at least 3)

### Insight 1: Technology drives the highest average sales but Furniture shows more variable profitability
From the box plots and group-by analysis, Technology category consistently shows higher median sales. However, Furniture has wider profit variance (including more negative outliers), indicating higher risk or higher shipping/cost sensitivity.

### Insight 2: Discount has a clear negative relationship with Profit
The scatter plot and correlation matrix show that higher discounts are associated with lower (often negative) profits. This suggests the current discount strategy may be eroding margins, especially in certain sub-categories.

### Insight 3: West and Ontario regions dominate both sales volume and profitability
Regional analysis reveals West and Ontario as the strongest performing regions in both total sales and total profit. Atlantic and Yukon lag significantly. Resource allocation (marketing, inventory) should prioritize high-performing regions while investigating under-performers.

### Additional Insight: Clear seasonality and growth trend
Monthly trend shows increasing sales over 2009–2012 with noticeable peaks, likely related to year-end or back-to-school periods. This supports time-aware inventory planning.

## 10. Conclusions & Business Recommendations

**Conclusions:**
- The cleaned Superstore dataset is ready for modeling and shows rich structure across categories, regions, and time.
- Sales and Profit are moderately correlated; Discount negatively impacts Profit.
- Customer Segment and Product Category are strong differentiators of performance.

**Business Recommendations:**
1. **Review Discount Policy** – High discounts are linked to profit loss. Consider targeted, limited-time discounts rather than broad reductions.
2. **Focus on Technology & High-Margin Sub-categories** – Allocate more inventory and marketing budget here.
3. **Regional Strategy** – Double down on West & Ontario; investigate root causes of weaker regions (shipping cost, product mix, competition).
4. **Seasonal Planning** – Use the clear monthly patterns for better demand forecasting and staffing.
5. **Customer Segment Targeting** – Corporate and Consumer segments show different behaviors; tailor promotions accordingly.

These insights form a solid foundation for the Sales Prediction Model in Task 3.